# Chỉ số dẫn trước: lúc 10h sáng đã biết hôm nay đông hay vắng?

Bộ `bike-sharing-uci`, `hour.csv`: 17 379 giờ của Capital Bikeshare (Washington DC), 01/01/2011 tới
31/12/2012. Cột dùng trong bài: `dteday` (ngày), `hr` (giờ 0-23), `workingday` (1 = ngày làm việc, 0 = cuối
tuần hoặc ngày lễ), `cnt` (lượt thuê trong giờ đó). Một số giờ không có dòng nào (hệ thống không ghi nhận):
coi như 0 lượt.

**Bối cảnh.** Trưởng ca vận hành nói: "Tổng lượt thuê cả ngày là KPI của tôi, nhưng tối mới biết thì đã muộn.
Lúc 10h sáng tôi phải quyết định có gọi thêm xe tải điều xe cho buổi chiều không. Có con số nào lúc 10h cho tôi
biết hôm nay sẽ đông hay vắng không?"

Tổng lượt cả ngày là **lagging indicator**. Bạn kiểm một ứng viên **leading indicator**: lượt thuê trước 10h
sáng. Một leading indicator chỉ có giá trị nếu nó **dự báo được** kết quả, nên bạn biến nó thành dự báo
(nowcast) rồi đo sai số.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa dữ liệu đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/01-analyst-mindset/exercises/basic/03-morning-leading-indicator -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc hour.csv
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

BIKE = ensure("bike-sharing-uci")
hour = pd.read_csv(BIKE / "hour.csv")
print(hour.shape, hour.dteday.min(), "→", hour.dteday.max())
hour[["dteday", "hr", "workingday", "cnt"]].head()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

## 1. Lượt buổi sáng và tổng cả ngày

`daily_signal(hour, cutoff=10) -> pd.DataFrame`: mỗi ngày một dòng, index là ngày (`DatetimeIndex` tên
`dteday`, tăng dần), ba cột theo thứ tự:

- `morning`: tổng `cnt` của các giờ `hr < cutoff` trong ngày (ngày không có giờ nào trước `cutoff` thì `0`);
- `total`: tổng `cnt` cả ngày;
- `workingday`: giá trị `workingday` của ngày đó.

`dteday` trong file là chuỗi, đổi sang ngày bằng `pd.to_datetime`.

In [ ]:
def daily_signal(hour: pd.DataFrame, cutoff: int = 10) -> pd.DataFrame:
    """Mỗi ngày: morning (cnt các giờ hr < cutoff), total, workingday; index dteday (datetime)."""
    day = pd.to_datetime(hour["dteday"])
    morning = hour["cnt"].where(hour["hr"] < cutoff, 0).groupby(day).sum()
    total = hour["cnt"].groupby(day).sum()
    workingday = hour["workingday"].groupby(day).first()
    out = pd.DataFrame({"morning": morning, "total": total, "workingday": workingday})
    out.index.name = "dteday"
    return out

In [ ]:
# explore: buổi sáng và cả ngày, tách theo loại ngày
daily = daily_signal(hour)
fig, ax = plt.subplots(figsize=(7, 4.5))
for wd, color, label in [(1, "#4c78a8", "ngày làm việc"), (0, "#f58518", "cuối tuần / lễ")]:
    part = daily[daily.workingday == wd]
    ax.scatter(part.morning, part.total, s=10, alpha=0.6, color=color, label=label)
ax.set_xlabel("lượt thuê trước 10h")
ax.set_ylabel("lượt thuê cả ngày")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("Hai loại ngày nằm trên hai đường khác nhau", loc="left")
plt.show()
print(f"tương quan gộp: {daily.morning.corr(daily.total):.3f}")
print(daily.groupby("workingday").apply(lambda g: round(g.morning.corr(g.total), 3)))

## 2. Buổi sáng chiếm bao nhiêu phần của cả ngày

`morning_share(daily) -> pd.Series`: với mỗi giá trị `workingday`, tỉ lệ **tổng** `morning` trên **tổng**
`total` của các ngày thuộc loại đó (tỉ lệ của tổng, không phải trung bình các tỉ lệ từng ngày). Index là
`workingday` (tăng dần), tên Series `share`.

In [ ]:
def morning_share(daily: pd.DataFrame) -> pd.Series:
    """Tổng morning / tổng total theo workingday, tên "share"."""
    sums = daily.groupby("workingday")[["morning", "total"]].sum()
    return (sums["morning"] / sums["total"]).rename("share")

## 3. Dự báo lúc 10h và sai số

`nowcast(daily, share) -> pd.Series`: dự báo tổng lượt cả ngày bằng lượt buổi sáng chia cho tỉ lệ buổi sáng
của **đúng loại ngày** đó: `morning / share[workingday]`. Cùng index với `daily`, tên `forecast`.

`median_ape(actual, forecast) -> float`: trung vị của sai số tương đối tuyệt đối
`|forecast - actual| / actual`, bỏ các ngày `actual == 0`. Dùng trung vị chứ không dùng trung bình vì vài
ngày bão (tổng chỉ vài chục lượt) cho sai số hàng trăm phần trăm và kéo trung bình.

In [ ]:
def nowcast(daily: pd.DataFrame, share: pd.Series) -> pd.Series:
    """Dự báo tổng ngày = morning / share của loại ngày đó, tên "forecast"."""
    return (daily["morning"] / daily["workingday"].map(share)).rename("forecast")


def median_ape(actual: pd.Series, forecast: pd.Series) -> float:
    """Trung vị |forecast - actual| / actual, bỏ ngày actual == 0."""
    ok = actual != 0
    return float(((forecast[ok] - actual[ok]).abs() / actual[ok]).median())

In [ ]:
# explore: tỉ lệ tách theo loại ngày so với một tỉ lệ chung cho mọi ngày
share = morning_share(daily)
print(share.round(4))
seg = nowcast(daily, share)
pooled = daily.morning.sum() / daily.total.sum()
one = nowcast(daily, pd.Series({0: pooled, 1: pooled}))
print(f"tỉ lệ chung: {pooled:.4f}")
print(f"sai số trung vị, tỉ lệ theo loại ngày: {median_ape(daily.total, seg):.1%}")
print(f"sai số trung vị, một tỉ lệ chung:     {median_ape(daily.total, one):.1%}")
ape = (seg - daily.total).abs() / daily.total
print(f"tỉ lệ ngày sai trong ±20%: {(ape <= 0.2).mean():.1%}")

In [ ]:
# explore: kiểm ngoài mẫu - lấy tỉ lệ từ năm 2011, dự báo năm 2012
train, test = daily[daily.index.year == 2011], daily[daily.index.year == 2012]
share_2011 = morning_share(train)
print(share_2011.round(4))
print(f"sai số trung vị 2012 với tỉ lệ 2011: {median_ape(test.total, nowcast(test, share_2011)):.1%}")

In [ ]:
# explore: những ngày dự báo sai nặng nhất - chuyện gì xảy ra?
worst = ape.nlargest(5).rename("ape").to_frame().join(daily)
worst["forecast"] = seg.loc[worst.index].round(0)
print(worst.round(2))

In [ ]:
# explore: đánh đổi giữa sớm và chính xác - cutoff 8h, 10h, 12h
for cutoff in [8, 10, 12]:
    d = daily_signal(hour, cutoff)
    print(f"trước {cutoff:>2}h: sai số trung vị {median_ape(d.total, nowcast(d, morning_share(d))):.1%}")

## 4. Đọc kết quả, rồi làm trắc nghiệm

Trả lời cho trưởng ca bằng hai, ba câu:

1. Lượt thuê trước 10h có phải là leading indicator dùng được không, sai số điển hình bao nhiêu?
2. Vì sao phải dùng tỉ lệ riêng cho ngày làm việc và cuối tuần? Chuyện gì xảy ra nếu dùng một tỉ lệ chung?
3. Những ngày nào chỉ số này "không thấy trước"? Cần thêm tín hiệu gì cho những ngày đó (dự báo thời tiết)?
4. Đổi 10h sang 8h hay 12h thì được và mất gì? Ai quyết định giờ nào là đủ sớm?

Xong thì mở app, làm phần **trắc nghiệm** của bài.